# **Implementation**
## **The goal of the notebook**
---
---
The goal of this example is to provide an example for running calculations with ciderpress XC functionals. We do that by running a single point calculation on a TBLite optimized β-MnO2-Pt surface with OH adsorbed on it.

**Workflow**

1.   β-MnO2-Pt surface creation and relaxation (TBLite)
2.   Surface + adsorbate relaxation (TBLite)
3.   Single-point calculation (Ciderpress)
---
---

---
---
## **Installation of packages**

---

First thing first, we need to import the neccessary functions. We need ASE, TBLite, Ciderpress and its functionals, and GPAW (MixerFull functional). We also import numpy

---
---

In [2]:
##################
##-dependencies-##
##################

##-ASE
from ase import Atom, Atoms
from ase.build import molecule, surface, add_adsorbate
from ase.io import read, write
from ase.visualize import view
from ase.optimize import BFGS
from ase.constraints import FixAtoms

##-TBLite
from tblite.ase import TBLite

##-Ciderpress
from ciderpress.gpaw.calculator import CiderGPAW, get_cider_functional

##-GPAW
from gpaw import GPAW, PW, MixerFull #-also needed for Ciderpress

##-general
import numpy as np

---
---
## **β-MnO2-Pt surface**

---

### **1.   β-MnO2-Pt surface creation and relaxation (TBLite)**
2.   Surface + adsorbate relaxation (TBLite)
3.   Single-point calculation (Ciderpress)
---

Here we download the unit cell for β-MnO2 and turn it into a
surface. Using the ASE surface function we create a β-MnO2(110) slab and replace one of the surface manganese atoms with a platinum atom, creating the β-MnO2-Pt surface. After succesfully creating the surface, we use TBLite calculator and BFGS optimizer for relaxing the surface. For simplicity sake we assume charge = 0, multiplicity = 1, and we don't use spin_polarization.

---
---

In [3]:
#################
##-MnO2 object-##
#################

##-Get the initial .cif file from materials project:
!wget https://raw.githubusercontent.com/doublelayer/test_models/refs/heads/main/Oxides/MnO2_beta.cif
mno2 = read('MnO2_beta.cif')

view(mno2, viewer='x3d')

--2026-03-13 09:19:50--  https://raw.githubusercontent.com/doublelayer/test_models/refs/heads/main/Oxides/MnO2_beta.cif
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1045 (1.0K) [text/plain]
Saving to: ‘MnO2_beta.cif’

MnO2_beta.cif       100%[===================>]   1.02K  --.-KB/s    in 0s      

2026-03-13 09:19:50 (57.3 MB/s) - ‘MnO2_beta.cif’ saved [1045/1045]



In [4]:
####################
##-MnO2-Pt object-##
####################

##-Creating the surface
slab = surface(mno2, (1,1,0), layers=2)
slab = slab.repeat((1,2,1))

##-Using the wrap() function to move topmost oxygens to the bottom
z = slab.positions[[a.index for a in slab if a.symbol == "Mn"], 2]
dz = (abs(z.max() - z.min()))
slab.cell[2,2] = 2*dz
slab.set_cell(slab.cell)
slab.positions[:,2] += 0.9*dz
slab.pbc=(True,True,True)
slab.wrap()

##-Replacing one of the manganese atoms with a platinum atom
slab += Atom("Pt", slab.positions[18])
del slab[18]

##-Setting the surface periodic in x and y coordinates
slab.pbc=(True,True,False)
slab.center(axis=2, vacuum=1.5*dz)
slab.positions[:,2] += -dz

view(slab, viewer='x3d')

In [5]:
####################################
##-optimization of MnO2-Pt(1,1,0)-##
####################################

slab.calc = TBLite(
    method='GFN1-xTB',
    charge=0,
    multiplicity=1,
    spin_polarization=0,
    max_iterations=350,
)

opt = BFGS(slab, trajectory='slab.traj', logfile='slab.log')

opt.run(fmax=0.05)

write('slab.xyz', slab)

------------------------------------------------------------
  cycle        total energy    energy error   density error
------------------------------------------------------------
      1     -57.01389673002  -5.7043673E+01   1.0278043E+00
      2      73.80553779921   1.3081943E+02   2.0333183E+00
      3      112.3509139215   3.8545376E+01   1.6587252E+00
      4      61.54302183692  -5.0807892E+01   1.2550260E+00
      5      2.880861727765  -5.8662160E+01   1.2294727E+00
      6     -8.051490250875  -1.0932352E+01   1.0784327E+00
      7     -37.35604190126  -2.9304552E+01   9.6672754E-01
      8     -25.17405772926   1.2181984E+01   1.0256928E+00
      9     -35.88799586394  -1.0713938E+01   1.0466447E+00
     10     -14.40165268331   2.1486343E+01   1.0255362E+00
     11     -52.05289591614  -3.7651243E+01   7.0778746E-01
     12     -63.22815440105  -1.1175258E+01   6.4758699E-01
     13     -80.53777231774  -1.7309618E+01   4.2102566E-01
     14     -82.70806999084  -2.170297

---
---
## **β-MnO2-Pt + OH surface**

---
1.   β-MnO2-Pt surface creation and relaxation (TBLite)
### **2.   Surface + adsorbate creation and relaxation (TBLite)**
3.   Single-point calculation (Ciderpress)
---

Here we use the previously optimized surface and ASE add_adsorbate function. We first locate the platinum atom, save it's position number, and finally add the adsorbate ontop of the platinum atom. Identically to before we optimize the surface + OH using TBLite.

---
---

In [6]:
###############
##-Adding OH-##
###############

slab_oh = read('slab.xyz')

oh = molecule('OH')
oh.rotate(90, 'y')

##-Adsorbent height
h = 2

##-Pt-OH
Pt_idx = [i for i, a in enumerate(slab_oh) if a.symbol == 'Pt'][0]
Pt_pos = slab_oh[Pt_idx].position[:2]

add_adsorbate(
    slab_oh,
    adsorbate=oh,
    height=h,
    position=Pt_pos,
)

In [7]:
#########################################
##-optimization of MnO2-Pt + OH(1,1,0)-##
#########################################

slab_oh.calc = TBLite(
    method='GFN1-xTB',
    charge=0,
    multiplicity=1,
    spin_polarization=0,
    max_iterations=350,
)

opt = BFGS(slab_oh, trajectory='slab_oh.traj', logfile='slab_oh.log')

opt.run(fmax=0.05)

write('slab_oh.xyz', slab_oh)
view(slab_oh, viewer='x3d')

Streaming output truncated to the last 5000 lines.
        [ 0.00000000e+00,  0.00000000e+00]]]), 'molecular-dipole': array([-10.05007798,  10.33450701,   3.46009214])}
{'bond-orders': array([[[ 0.00000000e+00,  6.20294872e-02],
        [ 1.46966044e-01,  3.46827651e-01],
        [ 2.14054995e-03,  3.45541470e-01],
        ...,
        [ 1.57467555e-03,  1.63324776e-03],
        [ 1.89830237e-03,  1.81780206e-02],
        [ 2.01707713e-02,  2.19559742e-04]],

       [[ 1.46966044e-01, -2.78432339e-04],
        [ 0.00000000e+00,  2.37683852e-03],
        [ 5.75913886e-04,  2.36771834e-03],
        ...,
        [ 1.25194375e-03,  4.04243203e-02],
        [ 8.70707272e-04,  2.30630751e-04],
        [ 3.11318455e-04,  7.06219044e-06]],

       [[ 2.14054995e-03,  1.62797916e-01],
        [ 5.75913886e-04,  1.43251142e-03],
        [ 0.00000000e+00,  1.49229721e-03],
        ...,
        [ 9.71625204e-04,  4.07083720e-02],
        [ 1.37532630e-03,  2.52282430e-04],
        [ 3.34288719e-04

/usr/local/lib/python3.12/dist-packages/ase/io/extxyz.py:318: UserWarning: Skipping unhashable information adsorbate_info
  warnings.warn('Skipping unhashable information '


---
---
## **Ciderpress calculation**

---
1.   β-MnO2-Pt surface creation and relaxation (TBLite)
2.   Surface + adsorbate creation and relaxation (TBLite)
### **3.   Single-point calculation (Ciderpress)**
---

Now we use the previously optimized β-MnO2-Pt + OH system to run a single-point calculation using the Ciderpress XC functionals. For the sake of time, we set the magnetic moments to 0 for all atoms. We also lowered the convergence criterium with the same intention.

---
---

In [8]:
#################################
##-ciderpress for OH adsorbate-##
#################################

##-Optimized structure .xyz
slab_oh_opt = read('slab_oh.xyz')

##-setting magmoms for each atom as 0 for the example with Ciderpress
slab_oh_opt.set_initial_magnetic_moments([0.0] * len(slab_oh_opt))

#lab_oh_opt.pbc = (True, True, True)

mlfunc = 'CIDER23X_SL_GGA.yaml'

xc = get_cider_functional(
    mlfunc,
    xmix=0.25,
    qmax=300,
    lambd=1.8,
    pasdw_ovlp_fit=True,
    use_paw=False,
)

slab_oh_opt.calc = CiderGPAW(
    h=0.20,
    xc=xc,
    mode='lcao',
    basis='dzp',
    txt='-',
    occupations={'name': 'fermi-dirac', 'width': 0.10},
    kpts={'size': (2, 2, 1)},
    convergence={'energy': 1e-4, 'density': 1e-3, 'eigenstates': 1e-6},
    mixer=MixerFull(beta=0.05, nmaxold=5, weight=50),
    spinpol=False,
)

etot = slab_oh_opt.get_potential_energy()


  ___ ___ ___ _ _ _  
 |   |   |_  | | | | 
 | | | | | . | | | | 
 |__ |  _|___|_____|  25.7.0
 |___|_|             

User:   ???@5884159f7f5d
Date:   Fri Mar 13 09:23:29 2026
Arch:   x86_64
Pid:    419
CWD:    /content
Python: 3.12.12
gpaw:   /usr/local/lib/python3.12/dist-packages/gpaw
_gpaw:  /usr/local/lib/python3.12/dist-packages/
        _gpaw.cpython-312-x86_64-linux-gnu.so
ase:    /usr/local/lib/python3.12/dist-packages/ase (version 3.27.0)
numpy:  /usr/local/lib/python3.12/dist-packages/numpy (version 2.0.2)
scipy:  /usr/local/lib/python3.12/dist-packages/scipy (version 1.16.3)
libxc:  5.1.7
units:  Angstrom and eV
cores: 1
OpenMP: False
OMP_NUM_THREADS: 1

Input parameters:
  basis: dzp
  convergence: {density: 0.001,
                eigenstates: 1e-06,
                energy: 0.0001}
  h: 0.2
  kpts: {size: (2, 2, 1)}
  mixer: {backend: pulay,
          beta: 0.05,
          method: fullspin,
          nmaxold: 5,
          weight: 50}
  mode: lcao
  occupations: {name: fer

/usr/local/lib/python3.12/dist-packages/ciderpress/gpaw/interp_paw.py:69: RuntimeWarning: invalid value encountered in divide
  temp /= r_g_new**2


XC parameters: CiderGGA with 2 nearest neighbor stencil

Memory estimate:
  Process memory now: 698.58 MiB
  Calculator: 169.01 MiB
    Density: 70.83 MiB
      Arrays: 12.70 MiB
      Localized functions: 53.28 MiB
      Mixer: 4.85 MiB
    Hamiltonian: 11.78 MiB
      Arrays: 8.31 MiB
      XC: 0.00 MiB
      Poisson: 0.00 MiB
      vbar: 3.48 MiB
    Wavefunctions: 86.39 MiB
      C [qnM]: 1.93 MiB
      S, T [2 x qmm]: 12.69 MiB
      P [aqMi]: 0.32 MiB
      BasisFunctions: 71.44 MiB
      Eigensolver: 0.00 MiB

Total number of cores used: 1

Number of atoms: 26
Number of atomic orbitals: 456
Number of bands in calculation: 139
Number of valence electrons: 224
Bands to converge: occupied

... initialized

Initializing position-dependent things.

Density initialized from atomic densities


/usr/local/lib/python3.12/dist-packages/ciderpress/dft/settings.py:1834: RuntimeWarning: invalid value encountered in power
  s = np.sqrt(sigma) / (b * rho ** (4.0 / 3) + 1e-16)
/usr/local/lib/python3.12/dist-packages/ciderpress/dft/baselines.py:44: RuntimeWarning: invalid value encountered in power
  e[:] += LDA_FACTOR * rho ** (4.0 / 3)
/usr/local/lib/python3.12/dist-packages/ciderpress/dft/baselines.py:45: RuntimeWarning: invalid value encountered in power
  dedx[0] += 4.0 / 3 * LDA_FACTOR * rho ** (1.0 / 3)
/usr/local/lib/python3.12/dist-packages/ciderpress/dft/baselines.py:76: RuntimeWarning: divide by zero encountered in divide
  dx = c / (2 * np.sqrt(s2))
/usr/local/lib/python3.12/dist-packages/ciderpress/dft/baselines.py:77: RuntimeWarning: invalid value encountered in divide
  chfx = (3 * x**2 + np.pi**2 * np.log(1 + x)) / (
/usr/local/lib/python3.12/dist-packages/ciderpress/dft/baselines.py:80: RuntimeWarning: invalid value encountered in divide
  dchfx = (
/usr/local/lib/pyt

     .--------------.  
    /|              |  
   / |              |  
  /  |              |  
 *   |              |  
 |   |              |  
 |   |              |  
 |   |              |  
 |   |              |  
 |   |  H  O        |  
 |   |  O    MnO    |  
 |  Mn              |  
 |   |    PtO       |  
 |Mn |O      O    O |  
 |   |    O         |  
 O   | O   Mn   O   |  
 |   |            Mn|  
 |   O   Mn   O     |  
 |   .----------MnO-.  
 |  /              /   
 O /              /    
 |/              /     
 *--------------*      

Atomic positions and initial magnetic moments

Positions:
   0 Mn    -0.031895    0.000001    3.161097    ( 0.0000,  0.0000,  0.0000)
   1 Mn     3.066083    1.451730    3.298566    ( 0.0000,  0.0000,  0.0000)
   2 O      0.031905    0.000000    5.253996    ( 0.0000,  0.0000,  0.0000)
   3 O      5.016819    1.443740    3.450797    ( 0.0000,  0.0000,  0.0000)
   4 O      3.092118   -0.000002    4.725806    ( 0.0000,  0.0000,  0.0000)
   5 O   

/usr/local/lib/python3.12/dist-packages/ciderpress/dft/settings.py:1834: RuntimeWarning: invalid value encountered in power
  s = np.sqrt(sigma) / (b * rho ** (4.0 / 3) + 1e-16)


iter:   2 09:24:08  -221.708091        c -0.96
iter:   3 09:24:26  -206.827043        c -1.07
iter:   4 09:24:45  -194.936520        c -1.21
iter:   5 09:25:03  -190.006997        c -1.40
iter:   6 09:25:20  -186.956579        c -1.46
iter:   7 09:25:39  -185.453836        c -1.53
iter:   8 09:25:57  -182.446866        c -1.57
iter:   9 09:26:15  -179.831888        c -1.68
iter:  10 09:26:34  -178.361685        c -1.79
iter:  11 09:26:51  -176.932022        c -1.87
iter:  12 09:27:11  -176.238595        c -2.06
iter:  13 09:27:29  -175.769184        c -2.20
iter:  14 09:27:47  -174.989653        c -2.29
iter:  15 09:28:06  -174.974597        c -2.45
iter:  16 09:28:23  -175.199986        c -2.49
iter:  17 09:28:41  -175.227254        c -2.63
iter:  18 09:29:00  -175.430773        c -2.76
iter:  19 09:29:17  -175.423289        c -2.90
iter:  20 09:29:35  -175.420294c       c -3.09c

Converged after 20 iterations.

Dipole moment: (-16.649818, -13.026464, 0.176402) |e|*Ang

Energy contrib